# Sound map: grow a library on Colab

One "island" of the layered-texture library (see `docs/handoff/2026-11-18_sound_map_handoff.md`). Each machine grows its own library
with a different `SEED`; `tools/sound_map/merge.py` combines them, then the full-drive check runs on the merged library.

**Run all.** If Colab disconnects, reconnect and **Run all** again: the library is saved to your Google Drive after every round and
`--resume` continues from it. CPU only (a GPU does not help).

In [ ]:
# How many CPU cores this runtime has (free: about 2). More cores = faster.
!nproc
import os; CORES = os.cpu_count()

In [ ]:
# Settings: use a SEED no other machine uses (VPS used 1; e.g. Mac 2, laptop 3, Colab 4).
SEED = 4
ROUNDS = 40
BRANCH = 'sound-map-search'
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/ev-sound-map'
os.makedirs(OUT, exist_ok=True)
LIBRARY = f'{OUT}/library.seed{SEED}.json'
print('library will be saved to', LIBRARY)

In [ ]:
# Get the code. A private repository needs a GitHub token (Settings > Developer settings > fine-grained token, read access to this repo).
import getpass, subprocess
REPO = '/content/Personalized-EV-Synthetic-Sound'
if not os.path.exists(REPO):
    token = getpass.getpass('GitHub token (leave empty if the repo is public): ')
    url = f'https://{token}@github.com/PhurichAmorn/Personalized-EV-Synthetic-Sound.git' if token else 'https://github.com/PhurichAmorn/Personalized-EV-Synthetic-Sound.git'
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', url, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '--ff-only'], check=True)
!git -C {REPO} log --oneline -1

In [ ]:
# Node 20 (the synth runs in Node) and the Python packages.
!node --version 2>/dev/null || true
!if ! node --version 2>/dev/null | grep -qE 'v(2[0-9])'; then curl -fsSL https://deb.nodesource.com/setup_20.x | bash - >/dev/null && apt-get install -y nodejs >/dev/null; fi
!node --version
!pip -q install -r {REPO}/tools/sound_map/requirements.txt
!cd {REPO}/desktop && npm ci --no-audit --no-fund --loglevel=error
!cd {REPO} && node tools/sound_map/build.mjs && echo bundles built

In [ ]:
# Grow the library (about 30-35 s per round on 16 cores; on 2 cores expect several minutes per round).
# Same settings as the other machines: layered textures, pulsing limit 0.2.
!cd {REPO} && SEED={SEED} WORKERS={CORES} LIMIT_F=0.2 LIBRARY={LIBRARY} python tools/sound_map/generate.py {ROUNDS} --voices stack --textures all --resume

Done when the last line shows `'gen': 40`. Copy `library.seedN.json` from Google Drive to the machine that merges, then:

```
python tools/sound_map/merge.py out/library.seed1.json out/library.seed4.json -o out/library.frame.json
LIMIT_F=0.2 LIBRARY=library.frame.json POOL=$PWD/tools/sound_map/out/frame_pool.json python tools/sound_map/drive.py library
```
The drive check can be split too: `drive.py library --part 1/2` on one machine, `--part 2/2` on another, copy both part files next to
the library, then `drive.py library --parts 2`.